In [28]:
import databento as db
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from zoneinfo import ZoneInfo


import sys
import os
cwd = os.getcwd()
root = cwd.split("BernsteinMartingaleNet")[0] + "BernsteinMartingaleNet"
if root not in sys.path:
    sys.path.append(root)
from lib.utils import is_full_nyse_day

In [29]:
file = "/home/jkim/coding_projects/BernsteinMartingaleNet/MarketData/bars/xnas-itch-20180501-20251125.ohlcv-1m.dbn"
output = db.DBNStore.from_file(file)

In [30]:
df_full = output.to_df()
df_full["date"] = df_full.index.date

In [ ]:
bars = []
for date, df in df_full.groupby("date"):
    if not is_full_nyse_day(date):
        continue
    date_str = date.strftime("%Y-%m-%d")
    open_time = datetime.datetime.strptime(date_str + " 09:30:00", "%Y-%m-%d %H:%M:%S").replace(tzinfo= ZoneInfo("America/New_York"))
    close_time = datetime.datetime.strptime(date_str + " 16:00:00", "%Y-%m-%d %H:%M:%S").replace(tzinfo= ZoneInfo("America/New_York"))
    df = df.copy()  # Create explicit copy to avoid SettingWithCopyWarning
    df["ts_event"] = pd.to_datetime(df.index)
    df['ts_event_est'] = df['ts_event'].dt.tz_convert('America/New_York')
    #print(df["ts_event_est"])
    df = df[(df["ts_event_est"] >= open_time) & (df["ts_event_est"] < close_time)]
    df["seconds_since_open"] = (df["ts_event_est"] - open_time).dt.total_seconds()
    bars.append(df)
    #print(df.shape)


In [ ]:
df